# 06 確率の校正としきい値 — 「70%」は本当に70%か

## このノートの目的
- モデルが出す「大雨になる確率」が、**本当の確率とずれる**ことを確かめる（`class_weight` の副作用）。
- **校正曲線**（信頼度曲線）の読み方と、**Platt 補正**での直し方を知る。
- **Brier スコア**で、確率の良さを測る。
- **しきい値**を、見逃しと空振りの**コスト**から決める考え方を知る。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **校正（キャリブレーション）** | 予測確率が、**実際の頻度と合っている**こと。「確率0.3と予測した標本のうち、本当に3割が陽性」なら、校正されている。 |
| **校正曲線** | 予測確率の帯ごとに、**予測の平均**と**実際の陽性率**を並べた図。対角線に乗れば、校正されている。 |
| **class_weight** | 陽性を重く扱う設定。**陽性を拾いやすくなる**が、確率は**実際より大きく**出る。 |
| **Platt 補正** | 予測確率（のロジット）を、**ロジスティック回帰**で実際の頻度に直す方法。順位はほとんど変わらず、確率の「値」だけ直る。 |
| **Brier スコア** | 予測確率と結果（0/1）の**二乗誤差の平均**。小さいほど良い。「いつも陽性率を返す」予測と比べて評価する。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

In [ ]:
# 目的: ExtraTrees の保持予測を、class_weight あり／なしの2通りで作る（leave-one-year-out、木は100本、3シードの平均）
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
import run_phase2 as p2

x = ds.build("naha"); f = ds.features("naha", change=False); target = "y20_region"
use = (x.year.isin(range(2010, 2025)) & x.wind_ok).to_numpy()
base_cfg = {**p2.BASE, "min_samples_leaf": 20, "n_estimators": 100, "change": False}
def make(cw):
    cfg = {**base_cfg, "class_weight": cw}
    return np.mean([p2.oof(x, f, target, "et", cfg, s) for s in range(3)], axis=0)
P = {"class_weight あり": make("balanced_subsample"), "class_weight なし": make(None)}
y = x.loc[use, target].to_numpy()
print(f"標本 {len(y)}、陽性 {y.sum()}、陽性率 {y.mean():.3f}")
for k, p in P.items():
    print(f"{k:16s}: 予測確率の平均 {p[use].mean():.3f}（実際の陽性率 {y.mean():.3f}）  PR-AUC {average_precision_score(y, p[use]):.3f}")

**出力の見方**: 「**class_weight あり**」は、予測確率の**平均**が実際の陽性率（約0.035）より**ずっと大きい**（0.3前後）はずです。陽性を重く扱ったぶん、確率が膨らんでいます。
一方、**PR-AUC（順位の良さ）は 0.09 と 0.08 で、差は小さい**です。確率の「値」はずれても、**順位**は保たれています。

## 1. 校正曲線

予測確率を**5つの帯**（低い順に20%ずつ）に分けて、帯ごとの**予測の平均**と**実際の陽性率**を比べます。

In [ ]:
# 目的: 校正曲線（予測確率の5分割 × 実際の陽性率）を描く
def reliability(p, y, bins=5):
    q = pd.qcut(pd.Series(p).rank(method="first"), bins, labels=False)
    return pd.DataFrame({"予測の平均": [p[q == k].mean() for k in range(bins)], "実際の陽性率": [y[(q == k).to_numpy()].mean() for k in range(bins)],
                         "標本数": [int((q == k).sum()) for k in range(bins)]})

fig, ax = plt.subplots(figsize=(5, 4.5))
for k, p in P.items():
    r = reliability(p[use], y)
    ax.plot(r["予測の平均"], r["実際の陽性率"], marker="o", label=k)
    print(k); print(r.round(3).T, "\n")
ax.plot([0, 0.6], [0, 0.6], "k--", lw=0.8, label="完全に校正")
ax.set_xlabel("予測の平均"); ax.set_ylabel("実際の陽性率"); ax.legend(); plt.show()

**出力の見方**: 点が**対角線（点線）より下**にあれば、「予測確率が**高すぎる**」という意味です。
**class_weight あり**は、**大きく下**にずれています（例: 予測の平均 0.56 の帯で、実際は 0.07）。ただし、**右上がり**（予測が高い帯ほど、実際の陽性率も高い）なので、**順位としては使える**ことが分かります。
`class_weight なし` は、**ほぼ対角線に乗っています**（予測の平均 0.008〜0.077 に対して、実際は 0.005〜0.073）。**確率としては正直**です。
ただし、確率の値が**小さい**（最大でも0.08ほど）ので、「0.5 以上なら警戒」のような**固定のしきい値**では、ほとんど警戒が出ません。**順位で警戒の割合を決める**（→ 3.）ほうが使いやすいです。

## 2. Platt 補正で、確率の値を直す

`class_weight あり` の予測確率を、**ロジスティック回帰**で実際の頻度に直します。
**他の年のデータで補正の式を学習し、その年に当てはめる**（leave-one-year-out）ので、評価に学習データの情報が漏れません。

In [ ]:
# 目的: Platt 補正（予測確率のロジット → ロジスティック回帰）を、年ごとに保持して当てはめる
def platt(p, x, mask):
    z = np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4))).reshape(-1, 1)    # ロジット（確率を -∞〜∞ に引き伸ばす）
    out = np.full(len(p), np.nan)
    for yr in range(2010, 2025):
        te = (x.year == yr).to_numpy()
        tr = (x.year.isin(range(2010, 2025)) & (x.year != yr)).to_numpy() & mask
        out[te] = LogisticRegression(C=1e6).fit(z[tr], x.loc[tr, target]).predict_proba(z[te])[:, 1]
    return out

pw = P["class_weight あり"]
pc = platt(pw, x, x.wind_ok.to_numpy())
rows = {"補正なし（class_weight あり）": pw[use], "Platt 補正あり": pc[use], "class_weight なし": P["class_weight なし"][use],
        "常に陽性率を返す": np.full(len(y), y.mean())}
pd.DataFrame({k: {"Brier スコア": brier_score_loss(y, v), "PR-AUC": average_precision_score(y, v) if np.std(v) > 0 else y.mean()} for k, v in rows.items()}).T.round(4)

**出力の見方**
- **Brier スコア**は、小さいほど良い指標です。**補正なし**（class_weight あり）は、確率が膨らんでいるので**非常に悪い**（約0.13）。**Platt 補正**で、約0.033まで改善します。
- ただし、**「常に陽性率を返す」**（何も知らない予測）の Brier スコアも約0.034で、**補正後とほぼ同じ**です。
  つまり、確率の値は「それらしく」直りましたが、**そもそも当てる力が弱い**ため、Brier スコアでは改善が見えません。
- 実は、`class_weight なし` の Brier スコア（約0.033）も、Platt 補正後と**同じ**です。陽性を重くしなければ、**最初から校正された確率**が出ます。
- PR-AUC は、補正で少し下がります（0.092 → 0.079）。年ごとに別の補正式を当てるため、**年をまたいだ順位が少し入れ替わる**からです。1つの式で全体を補正すれば、順位は変わりません。

## 3. しきい値を決める — 見逃しと空振りのコスト

「大雨かもしれない」と**警戒を出す**かどうかは、**確率のしきい値**で決めます。警戒を広げると、見逃しは減りますが、空振りが増えます。

In [ ]:
# 目的: 予測確率の上位 q% を警戒したときの、適合率・再現率・警戒の回数を表にする
p = pw[use]
rows = []
for q in (2, 5, 10, 20, 30, 50):
    thr = np.percentile(p, 100 - q)
    fl = p >= thr
    tp, fp, fn = int((fl & (y == 1)).sum()), int((fl & (y == 0)).sum()), int((~fl & (y == 1)).sum())
    rows.append({"警戒の割合(%)": q, "警戒の標本数": int(fl.sum()), "適合率": tp / max(tp + fp, 1), "再現率": tp / (tp + fn),
                 "見逃し（FN）": fn, "空振り（FP）": fp})
tab = pd.DataFrame(rows).set_index("警戒の割合(%)")
tab.round(3)

**出力の見方**: 警戒の割合を増やすと、**再現率（拾えた割合）は上がり、適合率（当たった割合）は下がります**。
どこで止めるかは、**数字では決まらず、「見逃し1回」と「空振り1回」のどちらが痛いか**で決まります。

### コストから考える
たとえば、「**見逃し1回の損失が、空振り1回の損失の `k` 倍**」と決めると、**損失の合計が最小になる警戒の割合**が見つかります。

In [ ]:
# 目的: 見逃しのコストが空振りの k 倍のとき、損失（見逃し×k ＋ 空振り）が最小になる警戒の割合を探す
k = 10                                           # ← 演習: 3, 30, 100 に変えてみよう（見逃しがどれだけ痛いか）
qs = np.arange(1, 100)
loss = []
for q in qs:
    fl = p >= np.percentile(p, 100 - q)
    fn = int((~fl & (y == 1)).sum()); fp = int((fl & (y == 0)).sum())
    loss.append(k * fn + fp)
best = int(qs[int(np.argmin(loss))])
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(qs, loss); ax.axvline(best, color="r", ls="--"); ax.set_xlabel("警戒の割合（%）"); ax.set_ylabel(f"損失（見逃し×{k} ＋ 空振り）"); plt.show()
print(f"k = {k} のとき、損失が最小になる警戒の割合: 約 {best}%（警戒なしの損失 {k * int((y == 1).sum())}、全部警戒の損失 {int((y == 0).sum())}）")

**出力の見方**: `k` が大きい（見逃しが痛い）ほど、**警戒を広げる**のが得になります。`k` が小さいと、**ほとんど警戒しない**ほうが損失が小さくなります。
このモデルは**当てる力が弱い**ので、`k` が小さいと、警戒を出すこと自体が割に合いません。

## まとめ
- `class_weight` は**陽性を拾いやすく**する代わりに、**確率の値を膨らませる**。値を使うなら**校正**（Platt 補正）が必要。
- **校正曲線**で、確率の値が信用できるかを確認する。**順位**として使うだけなら、校正は不要。
- **Brier スコア**は、「常に陽性率」と比べて評価する。当てる力が弱いと、差が見えない。
- **しきい値**は、**見逃しと空振りのコストの比**で決める。モデルの精度だけでは決まらない。

## 演習
1. **1.** の `bins=5` を `bins=10` に変えてください（観察点: 点が増えると、曲線はどう見えるか。標本数が減ってがたつくか）。
2. **3.** の `k = 10` を `3`、`30`、`100` に変えて、最小になる警戒の割合を記録してください（観察点: `k` が大きくなるほど、警戒の割合は増えるか）。
3. **1.** の `make("balanced_subsample")` を `make("balanced")`（`class_weight="balanced"`）に変えて、予測確率の平均がどう変わるか見てください。
4. **2.** の Platt 補正を、`pw` ではなく `class_weight なし` の予測（`P["class_weight なし"]`）に当てて、Brier スコアがどう変わるか比べてください。